# Patcher Node — Unit Tests

Runs Loader → Planner → Patcher over **every** operation in the plan and
dumps each fragment as YAML. One LLM call per `create`/`update` op;
`keep` and `discard` short-circuit without calling the LLM.

**Pre-requisites:** files under `data/inputs/`, `OPENAI_API_KEY` in `.env`,
Qdrant up (Patcher degrades gracefully if a collection is missing).

In [1]:
# Step 1 — Imports + paths
#
# Test fixtures (which spec, rules, legacy to use) live in
# openapi_generator.config.paths, so the notebook stays free of glob magic
# and ad-hoc strings. Edit paths.py to switch fixtures.

import json
from pathlib import Path

import yaml

from openapi_generator.config import get_logger
from openapi_generator.config.paths import (
    ROOT,
    TEST_LEGACY_PATH,
    TEST_RULES_PATH,
    TEST_SPEC_PATH,
)
from openapi_generator.nodes.loader import loader_node
from openapi_generator.nodes.planner import planner_node
from openapi_generator.nodes.patcher import patcher_node

logger = get_logger(__name__)

REPO_ROOT   = Path(ROOT).resolve()
SPEC_PATH   = Path(TEST_SPEC_PATH).resolve()
RULES_PATH  = Path(TEST_RULES_PATH).resolve()
LEGACY_PATH = Path(TEST_LEGACY_PATH).resolve()

assert SPEC_PATH.is_file(),  f"Spec not found at {SPEC_PATH}"
assert RULES_PATH.is_file(), f"Rules not found at {RULES_PATH}"
# Legacy is optional — Patcher must work without it too.
legacy_present = LEGACY_PATH.is_file()

logger.info(f"Spec   : {SPEC_PATH.name}")
logger.info(f"Rules  : {RULES_PATH.name}")
logger.info(f"Legacy : {LEGACY_PATH.name if legacy_present else '(none)'}")

/home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_generator/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-26 23:06:57 [INFO] __main__: Spec   : 28532-i00.md
2026-05-26 23:06:57 [INFO] __main__: Rules  : rules_bank_28532-i00_full_20260427_214528.json
2026-05-26 23:06:57 [INFO] __main__: Legacy : rel_17_TS28532_ProvMnS.yaml


In [2]:
# Step 2 — Build a realistic state with Loader + Planner first

with open(RULES_PATH, "r", encoding="utf-8") as f:
    rules_bank = json.load(f)

legacy_openapi = None
if legacy_present:
    with open(LEGACY_PATH, "r", encoding="utf-8") as f:
        legacy_openapi = yaml.safe_load(f)

loader_state = {
    "spec_doc_path": str(SPEC_PATH),
    "rules_bank": rules_bank,
    "legacy_openapi": legacy_openapi,
}
loader_out = loader_node(loader_state)
planner_out = planner_node({**loader_state, **loader_out})
ops = planner_out["operations_plan"]

logger.info(f"Planner produced {len(ops)} operation(s).")
for i, op in enumerate(ops[:5]):
    logger.info(
        f"  {i}. [{op['priority']:>6}] {op['action']:>6} {op['method'].upper():>6} "
        f"{op['path']}  (rules={op['source_rule_ids']})"
    )

2026-05-26 23:06:57 [INFO] openapi_generator.nodes.loader: Loader → parsed 696 sections from 28532-i00.md (excluded 1 symbolic-title section(s))
2026-05-26 23:06:57 [INFO] openapi_generator.nodes.loader: Loader → rules_bank: 240 rule(s); legacy_openapi: present
2026-05-26 23:06:57 [INFO] openapi_generator.nodes.loader: Loader → seeding final_openapi from legacy (1 path(s), 16 schema(s))
2026-05-26 23:06:57 [INFO] openapi_generator.nodes.planner: Planner Node started (2 passes + gap check).
2026-05-26 23:06:57 [INFO] openapi_generator.config.llm_config: Default LLM ready: model=gpt-4.1-mini temperature=0.0
2026-05-26 23:06:57 [INFO] openapi_generator.nodes.planner: Planner Pass 1 → 4 legacy operation(s) to review.
2026-05-26 23:06:58 [INFO] openapi_generator.rag.qdrant_factory: Connecting to Qdrant at localhost:6333
2026-05-26 23:06:58 [INFO] openapi_generator.config.hardware: Embedding device: cuda
2026-05-26 23:06:58 [INFO] openapi_generator.rag.qdrant_factory: Loading embeddings: sen

In [3]:
# Step 3 — Run the Patcher on every operation of the plan

fragments = []
for i in range(len(ops)):
    state = {
        **loader_state,
        **loader_out,
        "operations_plan": ops,
        "current_op_idx": i,
        "op_iteration_count": 0,
    }
    out = patcher_node(state)
    frag = out["current_fragment"]
    fragments.append(frag)
    logger.info(
        f"  [{i+1}/{len(ops)}] {ops[i]['action']:>7} "
        f"{frag['method'].upper():>6} {frag['path']}  "
        f"(paths={len(frag.get('paths') or {})}, "
        f"schemas={len(((frag.get('components') or {}).get('schemas') or {}))})"
    )

2026-05-26 23:09:26 [INFO] openapi_generator.nodes.patcher: Patcher → op 1/33 (PUT /{className}={id}) action=update attempt=1
2026-05-26 23:09:26 [INFO] openapi_generator.rag.retriever: RAG retrieved 4 chunks for query='PUT /{className}={id} — className — id'
2026-05-26 23:09:26 [INFO] openapi_generator.nodes.patcher: Patcher → 3GPP RAG returned 4 chunk(s)
2026-05-26 23:09:26 [INFO] openapi_generator.nodes.patcher: Patcher → OpenAPI reference RAG returned chunks
2026-05-26 23:09:48 [INFO] openapi_generator.nodes.patcher: Patcher → produced fragment for PUT /{className}={id}: 1 path block(s), 0 new schema(s)
2026-05-26 23:09:48 [INFO] __main__:   [1/33]  update    PUT /{className}={id}  (paths=1, schemas=0)
2026-05-26 23:09:48 [INFO] openapi_generator.nodes.patcher: Patcher → op 2/33 (GET /{className}={id}) action=update attempt=1
2026-05-26 23:09:48 [INFO] openapi_generator.rag.retriever: RAG retrieved 4 chunks for query='GET /{className}={id} — className — id'
2026-05-26 23:09:48 [INF

In [4]:
# Step 4 — Inspect the first non-empty fragment as YAML

frag = next((f for f in fragments if (f.get("paths") or {})), fragments[0])
print(yaml.safe_dump(
    {"paths": frag.get("paths") or {}, "components": frag.get("components") or {}},
    sort_keys=False, allow_unicode=True,
))

paths:
  /{className}={id}:
    put:
      summary: Replaces a complete single resource or creates it if it does not exist
      description: With HTTP PUT a complete resource is replaced or created if it
        does not exist. The target resource is identified by the target URI.
      operationId: replaceOrCreateResource
      parameters:
      - name: className
        in: path
        required: true
        schema:
          type: string
        description: The managed object class name in the resource path
      - name: id
        in: path
        required: true
        schema:
          type: string
        description: The managed object instance identifier in the resource path
      - name: scope
        in: query
        required: false
        schema:
          type: string
        style: form
        explode: true
        description: Scope parameter
      - name: attributes
        in: query
        required: false
        schema:
          type: array
          items:
   

In [5]:
# Step 5 — Persist every Patcher fragment to data/outputs/test_patcher/

from datetime import datetime
from openapi_generator.config.paths import OUTPUTS_TEST_PATCHER_DIR

OUT_DIR = Path(OUTPUTS_TEST_PATCHER_DIR).resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# One YAML file with every non-empty fragment under paths/components.
all_paths = {}
all_schemas = {}
for f in fragments:
    for p, item in (f.get("paths") or {}).items():
        all_paths.setdefault(p, {}).update(item if isinstance(item, dict) else {})
    for name, schema in ((f.get("components") or {}).get("schemas") or {}).items():
        all_schemas.setdefault(name, schema)

fragments_path = OUT_DIR / f"fragments_{ts}.yaml"
fragments_path.write_text(
    yaml.safe_dump(
        {"paths": all_paths, "components": {"schemas": all_schemas}},
        sort_keys=False, allow_unicode=True,
    ),
    encoding="utf-8",
)
logger.info(f"Wrote merged fragments YAML: {fragments_path}")

# Companion JSON: one record per op with action + sizes (handy for diffing runs).
summary_path = OUT_DIR / f"summary_{ts}.json"
summary_path.write_text(
    json.dumps(
        [
            {
                "idx": i,
                "action": ops[i]["action"],
                "method": frag.get("method"),
                "path": frag.get("path"),
                "paths_count": len(frag.get("paths") or {}),
                "schemas_count": len((frag.get("components") or {}).get("schemas") or {}),
                "source_rule_ids": ops[i]["source_rule_ids"],
            }
            for i, frag in enumerate(fragments)
        ],
        indent=2, ensure_ascii=False,
    ),
    encoding="utf-8",
)
logger.info(f"Wrote per-op summary JSON : {summary_path}")

2026-05-26 23:26:19 [INFO] __main__: Wrote merged fragments YAML: /home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_generator/data/outputs/test_patcher/fragments_20260526_232619.yaml
2026-05-26 23:26:19 [INFO] __main__: Wrote per-op summary JSON : /home/arimatea/Documents/Pessoal/Mestrado/0-Mestrado_Unicamp_2025/5-Projeto_mestrado_ericsson/openapi_multiagents/workspace/openapi_generator/data/outputs/test_patcher/summary_20260526_232619.json
